# TNT hybrid V3 — M1 pretraining gate

This isolated experiment tests the diagnostic-driven V3 architecture on **M1 synthetic pretraining → Gold-train fine-tuning**. It keeps the same 3,330 M1 pairs, seed, epoch budget, learning rates, and Gold-dev checkpoint selection used by TNT V1 and V2. **Gold-test is not loaded or evaluated.**

V3 changes only the architecture/supervision: replacement length is predicted separately from replacement content; length loss applies only to substitution/expansion positions; rare boundary actions use focal loss; and the word head receives French/entity/protection flags. Only reviewed safe surface forms can be hard-kept; French/entity status alone does not block a correction.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    candidates = [parent for parent in PROJECT_ROOT.parents if (parent / 'src').is_dir()]
    if not candidates:
        raise FileNotFoundError('Run from the PFE repository or one of its subdirectories.')
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from IPython.display import display

from src.modeling.tnt_hybrid_v3_m1_gate import (
    BENCHMARK_NAME, RESULTS_DIR, gate_decision, gate_prediction_comparison,
    gate_protocol, gate_summary, prepare_gate_data,
    prepare_gate_initialization, run_m1_gate,
)

print('Project:', PROJECT_ROOT)
print('Experiment:', BENCHMARK_NAME)
print('Results:', RESULTS_DIR.resolve())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Project: C:\Users\thioy\Desktop\School\PFE
Experiment: tnt_hybrid_v3_m1_gate_v1
Results: C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Recreate the controlled data selection

All retained synthetic corpora are loaded only to reproduce the frozen common-target selection and shared character vocabulary. The training cell below uses M1 only.

In [2]:
protocol = gate_protocol()
gold, selected, dataset_audit = prepare_gate_data(protocol)
display(pd.DataFrame([protocol.__dict__]))
display(dataset_audit)
print({split: len(frame) for split, frame in gold.items()})
print('Selected M1 rows:', len(selected['m1']))

,seed,matched_unique_rows,synthetic_epochs,gold_epochs,synthetic_learning_rate,gold_learning_rate,synthetic_batch_size,gold_batch_size,evaluation_batch_size,generated_eval_interval_epochs
0,2026,3330,20.0,20.0,0.0003,0.0001,64,32,64,5


,method,loaded_rows,unique_pair_rows,unique_target_rows,selected_rows,common_target_pool,seed,dev_input_overlap,dev_pair_overlap,test_input_overlap,test_pair_overlap,source_over_255_chars,target_over_255_chars
0,m1,17777,16377,16375,3330,3430,2026,0,0,0,0,8,6
1,m2,17777,16544,16375,3330,3430,2026,0,0,0,0,10,6
2,m3,17777,16384,16375,3330,3430,2026,0,0,0,0,8,6
3,m4,3433,3433,3430,3330,3430,2026,0,0,0,0,7,6
4,m6,17777,16427,16375,3330,3430,2026,0,0,0,0,6,6


{'train': 142, 'dev': 30, 'test': 29}
Selected M1 rows: 3330


## 2. Build or reuse the deterministic V3 initialization

Class weights are calculated from complete Gold-train labels before M1 optimization. The generated protocol records the counts, weights, thresholds, and loss settings. A separate experiment name prevents collision with the completed V2 run.

In [3]:
shared_initialization = prepare_gate_initialization(gold, selected, protocol)
architecture_protocol = json.loads(
    (RESULTS_DIR / 'architecture_protocol.json').read_text(encoding='utf-8')
)
print('Initialization:', shared_initialization.resolve())
display(pd.DataFrame([architecture_protocol['architecture']]).T.rename(columns={0: 'value'}))
print('Segment-length counts (edit positions only):')
display(pd.Series(architecture_protocol['edit_label_profile']['segment_length_counts']))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[TNT-V3] Created 1,783,707-parameter model with 1,499 lexical feature entries: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\shared_initialization\tnt_hybrid_v3_char_dadd682dc9cc
Initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\shared_initialization\tnt_hybrid_v3_char_dadd682dc9cc


,value
d_model,192
attention_heads,4
encoder_layers,3
feed_forward_size,768
dropout,0.1
max_position_embeddings,256
output_slots_per_source,16
operation_loss_weight,1.0
character_loss_weight,1.0
segment_length_loss_weight,1.0


Segment-length counts (edit positions only):


0       0
1     666
2     216
3      12
4       1
5       0
6       1
7       2
8       0
9       0
10      0
11      0
12      0
13      0
14      1
15      0
16      0
dtype: int64

## 3. Run M1 pretraining, then Gold fine-tuning

Set `RUN_M1_V3_GATE = True` when ready. `RERUN = False` reuses a completed run and resumes the latest incomplete checkpoint; set it to `True` only when intentionally starting a separate attempt. Training progress and generated Gold-dev evaluations remain visible in the notebook output.

In [4]:
RUN_M1_V3_GATE = True
RERUN = False

if RUN_M1_V3_GATE:
    m1_v3_record = run_m1_gate(
        gold, selected, shared_initialization, protocol, rerun=RERUN
    )
    display(pd.DataFrame([m1_v3_record.get('development_metrics', {})]))
else:
    print('Training is disabled. Set RUN_M1_V3_GATE = True and rerun this cell.')


[M7] START m1 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\shared_initialization\tnt_hybrid_v3_char_dadd682dc9cc


You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

[M7] Loaded 1,783,707 total parameters; training 1,783,707 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


Tokenizing TNT edit pairs:   0%|          | 0/3330 [00:00<?, ? examples/s]

...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridV3TransformerForConditionalGeneration.forward` and have been ignored: source_text, source_id, target_text. If source_text, source_id, target_text are not expected by `TntHybridV3TransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.19448584202682564
[compact benchmark] m1: continuous 20-epoch gold run complete; selected dev CER=0.19448584202682564


,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate
0,30,0.194486,0.691525,59.948595,0.0,2,69,70,0.028169,0.027778,0.027972,0.971831,0,0.0


## 4. Compare V3 with identity, TNT V1, and hybrid V2

These cells remain safe before training: the summary includes whichever matched checkpoints exist. The decision compares V3 with V2 and never opens Gold-test.

In [5]:
summary = gate_summary(gold['dev'])
display(summary)
display(gate_decision(summary))

,condition,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate
0,identity,30,0.233234,0.630508,56.052143,0.0,0,0,72,0.000000,0.000000,0.000000,0.000000,0,0.0
1,tnt_v1_m1,30,0.181073,0.667797,55.888125,0.0,2,67,70,0.028986,0.027778,0.028369,0.971014,0,0.0
2,tnt_hybrid_v2_m1,30,0.200447,0.661017,56.463451,0.0,2,76,70,0.025641,0.027778,0.026667,0.974359,0,0.0
3,tnt_hybrid_v3_m1,30,0.194486,0.691525,59.948595,0.0,2,69,70,0.028169,0.027778,0.027972,0.971831,0,0.0


{'ready': True,
 'v3_improves_v2_on_all_predeclared_criteria': False,
 'criteria': {'cer_improved_over_v2': True,
  'wer_not_worse_than_v2': False,
  'correction_f1_improved_over_v2': True,
  'overcorrection_reduced_from_v2': True,
  'at_least_one_true_correction': True},
 'selection_split': 'Gold-dev',
 'gold_test_used': False}

In [6]:
comparison = gate_prediction_comparison(gold['dev'])
prediction_columns = [column for column in comparison if column.startswith('prediction_')]
if prediction_columns:
    changed = comparison.loc[
        comparison[prediction_columns].ne(comparison['source_text'], axis=0).any(axis=1)
    ]
    display(changed.head(30))
else:
    print('No completed TNT prediction files were found.')

,source_id,source_text,reference,prediction_v1,prediction_v2,prediction_v3
0,gold:dev:10000,​ té dou wakhé dank,té du waxee ndànk,te du waxe dank,te du waxe daank,te du waxe dan
1,gold:dev:10003,cachet wam mo wara diekh. sokhna ak salimata l...,cachet wam moo wara jeex sokhna ak salimata lé...,cachet wam mo wara djeex. soxna ak salimata le...,cachet wam moo waara jeex. soxna ak salimata l...,cachet wam moo wara jex. sokh na ak salimata l...
2,gold:dev:10005,nimako diépé sakh da eupeu,ni ma ko jéppe sax da ëpp,nimako diépé sax da ëpë,nimako diépe sax da ëpë,nimako deépee sax da ëpë
3,gold:dev:10007,maman oumy mome rk lay bagna guis,maman oumy moom rek laay baña gis,maman oumy mome rk lay bañña gis,maman oumy moom rk lay baaa ggis,maman oumy moome rk laay ba aa gis
4,gold:dev:10008,mére bi mo sof ba paré reuy lamign,mère bi moo soof ba pare réy làmmiñ,mére bi mo sof ba pare rëy lamigñ,mére bi moo soof ba pare rëy lamie,mére bi mo soof ba pare rëy laamiññ
5,gold:dev:10013,franchement khady thiam nak dafa eupeul fi kou...,franchement khady thiam nak dafa ëppal fi kuy ...,franchement hady ciam nak dafa ëpël fi kuy sëy...,franceement xady cam nak dafa ëpël fi kuy sëy ...,fra nchement khady thiam nak dafa ëpël fi ky s...
6,gold:dev:10014,loutakh pere ndiouga di raamal vieux?alors que...,lu tax pere ndiouga di raamal vieux alors que ...,lutax per ndjouga di raamal viëëx?alors qu ñoo...,utax pere njñuga i raamal vjëx?aloors q ñoo bo...,lutax pere ndiou ga di raamal viëx?alors qé óo...
7,gold:dev:10020,ki di way baabel voix bi nèx na deh kanla,ki di woy baabel voix bi neex na de kan la,ki di way baabel voix bi nèx na dnx kanla,ki di waay baabel voix bi nèx na dne kanla,ki di waay baabel voix bi nèx na dex ka nla
8,gold:dev:10021,maman omou kiye rayé si série douko ray,maman omou kuy reye ci série du ko rey,maman omou kiy raye si sérin duko ray,maman omu kiyé raye si serjc duko ray,maman omou kiyée raye si série dé ko ray
9,gold:dev:10026,le meilleurs séries kou d'accord lalal j'aime ️,les meilleurs séries ku d'accord laalal j'aime,le meilleërs séries ku d'accord lalal j'aim ️,le meillërs seeries ku d'accord lalal j'aimé ️,le meillërs seerées ku d 'accord laalal j'aime ️


## Interpretation rule

Do not judge V3 from CER alone. A useful result should preserve or improve CER/WER while increasing true corrections and correction F1 and reducing overcorrection relative to V2. If V3 still mostly copies or overcorrects, inspect its operation, length, boundary, and word heads before increasing epochs.